# Lab 4 — Colour images, overfitting, and data augmentation

**Module:** 7144COMP — Deep Learning Concepts and Techniques  
**Week:** 4  
**Estimated time:** 180 minutes (most of this is training — see note in Section 5)

---

## Learning outcomes

By the end of this lab you should be able to:

1. Load and pre-process a colour image dataset (CIFAR-10) using `torchvision`, including correct per-channel normalisation.
2. Recognise **overfitting** by reading training and validation loss curves and explain what is happening mechanically.
3. Apply standard image data augmentation transforms (`RandomCrop`, `RandomHorizontalFlip`, `RandomRotation`) inside a `torchvision.transforms.Compose` pipeline.
4. Explain why augmentation only goes on the *training* transform pipeline and never on validation or test.
5. Quantitatively compare two models trained with and without augmentation on the same architecture, and articulate the practical difference.
6. Use class names (not integer indices) when reporting and plotting classification results.

## Prerequisites

- **Labs 1–3** completed. You should be comfortable with the PyTorch training-loop pattern (`run_epoch`, `EarlyStopping`) and the `nn.Module` style of model definition.
- The textbook *Applied Deep Learning* (Fergus & Chalmers), Chapter 4 — sections on convolutional networks and regularisation.
- Lecture 4: *Regularisation in deep learning — weight decay, dropout, and data augmentation*.

## The thread from last week

In Lab 3 we built a small CNN that hit ~99% on MNIST in 8 epochs. That's because MNIST is *visually simple* — same handwriting style, centred digits, no background. Real images don't behave that way. **CIFAR-10** is 32×32 colour photographs of ten everyday objects (cats, ships, frogs, trucks, …), with all the messiness real photos carry: cluttered backgrounds, varied poses, partial occlusions, weird lighting.

We will discover, by running an experiment, that the same architecture which sailed through MNIST **starts to overfit visibly on CIFAR-10 within a few epochs**. Then we will fix it with data augmentation — arguably the single most powerful regularisation technique in computer vision.

## Useful references

- [The CIFAR-10 dataset homepage](https://www.cs.toronto.edu/~kriz/cifar.html) — Krizhevsky's original page.
- [torchvision.transforms documentation](https://pytorch.org/vision/stable/transforms.html) — the full catalogue of augmentation transforms.
- Shorten, C. and Khoshgoftaar, T.M. (2019). *A survey on image data augmentation for deep learning*. Journal of Big Data 6(1) — a thorough survey.

---

## 1. The dataset

**CIFAR-10** is 60,000 32×32 colour images in 10 classes: 50,000 training and 10,000 test. Six thousand images per class, balanced. It was assembled by Krizhevsky, Nair and Hinton in 2009 and was the benchmark that drove a lot of mid-2010s computer-vision progress.

| Label | Class       | Label | Class      |
|-------|-------------|-------|------------|
| 0     | airplane    | 5     | dog        |
| 1     | automobile  | 6     | frog       |
| 2     | bird        | 7     | horse      |
| 3     | cat         | 8     | ship       |
| 4     | deer        | 9     | truck      |

**Why colour matters here.** CIFAR-10 images are *(3, 32, 32)* — three colour channels rather than the single grayscale channel of MNIST. That changes a few things:

- The first convolutional layer's input channels become **3, not 1**.
- Normalisation needs **per-channel** mean and standard deviation (different for R, G, B).
- The dataset is larger on disk (~170 MB) and slower to train on, even at the same spatial resolution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
RNG_SEED = 7144
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

In [ ]:
# CIFAR-10 per-channel mean and standard deviation, computed over the training set.
# These are the canonical values cited in every CIFAR-10 paper.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

CIFAR_CLASSES = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

# Baseline preprocessing: just convert to tensor and normalise. NO augmentation yet.
# This is also what we use for the validation and test sets.
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# First time you run this, torchvision downloads CIFAR-10 (~170 MB).
# After that it's cached in data/ and re-loads instantly.
train_full = datasets.CIFAR10(root="data", train=True, download=True, transform=eval_transform)
test_set = datasets.CIFAR10(root="data", train=False, download=True, transform=eval_transform)

print(f"Training samples: {len(train_full)}")
print(f"Test samples: {len(test_set)}")
img, lbl = train_full[0]
print(f"Image shape: {img.shape}  (C, H, W)")
print(f"Label: {lbl} ({CIFAR_CLASSES[lbl]})")

### 1.1 Train / validation / test split

Same discipline as Lab 3 — carve a 5,000-image validation set off the training set, leaving 45,000 for training. The 10,000 test images are held out until the end.

In [ ]:
VAL_SIZE = 5_000
TRAIN_SIZE = len(train_full) - VAL_SIZE

generator = torch.Generator().manual_seed(RNG_SEED)
train_set, val_set = random_split(train_full, [TRAIN_SIZE, VAL_SIZE], generator=generator)

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {TRAIN_SIZE} samples ({len(train_loader)} batches)")
print(f"Val:   {VAL_SIZE} samples ({len(val_loader)} batches)")
print(f"Test:  {len(test_set)} samples ({len(test_loader)} batches)")

### 1.2 What does the data actually look like?

Always look at your data. Here are 12 random training images with their class labels.

In [ ]:
def un_normalise(t: torch.Tensor) -> np.ndarray:
    """Reverse the normalisation so an image can be displayed naturally."""
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for ax, img, lbl in zip(axes.flat, xb[:12], yb[:12]):
    ax.imshow(un_normalise(img))
    ax.set_title(CIFAR_CLASSES[lbl.item()], fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

**What to observe.** Notice the variation *within* each class — different poses, different angles, different backgrounds, different lighting. This is what makes CIFAR-10 fundamentally harder than MNIST: the network must learn what makes a *cat* a cat in the abstract, not just one prototype shape repeated thousands of times.

## 2. A baseline CNN — and the overfitting it causes

We start with a deliberately modest CNN. Three conv blocks, no fancy tricks, no regularisation beyond a single dropout in the head. The point isn't peak accuracy — it's to **see overfitting happen** so we can fix it in the next section.

Architecture:

| Layer | Output shape | What it does |
|-------|--------------|--------------|
| Input | (3, 32, 32) | RGB image |
| Conv2d(3→32, 3×3, padding=1) + BN + ReLU | (32, 32, 32) | |
| MaxPool2d(2) | (32, 16, 16) | |
| Conv2d(32→64, 3×3, padding=1) + BN + ReLU | (64, 16, 16) | |
| MaxPool2d(2) | (64, 8, 8) | |
| Conv2d(64→128, 3×3, padding=1) + BN + ReLU | (128, 8, 8) | |
| MaxPool2d(2) | (128, 4, 4) | |
| Flatten | (2048,) | |
| Linear(2048 → 256) + ReLU + Dropout(0.4) | (256,) | |
| Linear(256 → 10) | (10,) | logits |

In [ ]:
class CIFARCNN(nn.Module):
    """A three-block CNN for 32x32x3 image classification."""

    def __init__(self, n_classes: int = 10, dropout: float = 0.4):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(256, n_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # (B, 32, 16, 16)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # (B, 64,  8,  8)
        x = self.pool(F.relu(self.bn3(self.conv3(x))))   # (B,128,  4,  4)
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


# Sanity check shape flow.
test_model = CIFARCNN().to(DEVICE)
with torch.no_grad():
    out = test_model(xb[:4].to(DEVICE))
n_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"Output shape: {out.shape}  (expected (4, 10))")
print(f"Trainable parameters: {n_params:,}")

## 3. Reusable training helpers

Same `run_epoch` and `EarlyStopping` we used in Labs 2 and 3. By this point you've seen this pattern several times — it really is the canonical PyTorch loop you'll use everywhere.

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 4, min_delta: float = 0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.epochs_without_improvement = 0
        self.best_state = None
        self.should_stop = False

    def step(self, current_loss: float, model: nn.Module) -> None:
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.epochs_without_improvement = 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.epochs_without_improvement += 1
            if self.epochs_without_improvement >= self.patience:
                self.should_stop = True

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(mode=is_train)
    total_loss, total_correct, total_samples = 0.0, 0, 0
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_samples += xb.size(0)
    return total_loss / total_samples, total_correct / total_samples


def train_model(model, train_loader, val_loader, max_epochs, lr=1e-3, patience=4, label="model"):
    """Standard train-with-early-stopping. Returns the history dict."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    es = EarlyStopping(patience=patience)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    print(f"\n=== Training {label} for up to {max_epochs} epochs ===")
    t0 = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader, criterion, optimizer=None)
        history["train_loss"].append(tl); history["train_acc"].append(ta)
        history["val_loss"].append(vl); history["val_acc"].append(va)
        print(f"epoch {epoch:>2d}  |  train loss {tl:.4f} acc {ta:.4f}  |  val loss {vl:.4f} acc {va:.4f}")
        es.step(vl, model)
        if es.should_stop:
            print(f"Early stopping at epoch {epoch}.")
            break
    es.restore_best(model)
    model.to(DEVICE)
    elapsed = time.perf_counter() - t0
    print(f"Done in {elapsed/60:.1f} min. Best val loss: {es.best_loss:.4f}")
    return history

## 4. Train the baseline (no augmentation)

> ⏱ **Time check.** Each epoch on a typical laptop CPU takes 60–90 seconds. **10 epochs ≈ 10–15 minutes.** On a GPU it's under a minute total. Now is a good moment to grab a coffee.

We deliberately train *without* augmentation here. Watch the gap between training and validation loss carefully.

In [ ]:
MAX_EPOCHS = 10

torch.manual_seed(RNG_SEED)
model_baseline = CIFARCNN().to(DEVICE)
history_baseline = train_model(
    model_baseline, train_loader, val_loader,
    max_epochs=MAX_EPOCHS, label="baseline (no augmentation)"
)

In [ ]:
def plot_history(h, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(h["train_loss"], label="train", marker="o")
    axes[0].plot(h["val_loss"], label="validation", marker="o")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy loss")
    axes[0].set_title(f"{title} — loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(h["train_acc"], label="train", marker="o")
    axes[1].plot(h["val_acc"], label="validation", marker="o")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
    axes[1].set_title(f"{title} — accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()


plot_history(history_baseline, "Baseline")

### 4.1 Reading the curves — overfitting in action

Look at the training-vs-validation curves you just plotted. The training loss should keep falling smoothly, but the **validation loss starts to flatten or even rise** while the training loss continues to drop. The corresponding accuracy plot tells the same story: training accuracy climbs, validation accuracy stalls.

This is **overfitting**. The model is no longer learning generalisable patterns — it's memorising idiosyncrasies of the *specific* 45,000 training images: a particular cat's particular ear position, a particular truck's particular angle. Those memorised details don't transfer to the 5,000 validation images the model has never seen.

**Why is overfitting happening?** Two reasons interacting:

1. **Model capacity** — our network has ~640,000 trainable parameters but only 45,000 training images. There's plenty of room to memorise.
2. **Limited data diversity** — each training image appears in *exactly the same form* every single epoch. The network sees the same pixels over and over, so it learns to associate those exact pixels with the label.

We could fix (1) by shrinking the network — but a small network would *underfit* and never reach decent accuracy. The better fix is to attack (2): **make the network see a different version of each image every epoch.** This is data augmentation.

---

## 5. Data augmentation

Data augmentation applies random transformations to each training image **on the fly during training**. The label stays the same; the image varies. Common transforms for natural images:

<img src="assets/augmentation.svg" alt="Data augmentation: original image with three transformed variants" width="880"/>

**Critical rule.** Augmentation goes on the **training transform pipeline only**. The validation and test sets must use the *original* preprocessing (just `ToTensor` + `Normalize`). If you augmented the test set you'd be measuring performance on a different distribution from the one you care about — and worse, performance would vary every time you ran the evaluation because the augmentations are random.

**Why augmentation works.**

- The network sees *effectively* more data: each training image yields infinitely many variants over many epochs.
- It enforces useful invariances: a horizontally-flipped cat is still a cat, so the network is forced to learn features that are invariant to horizontal flip rather than features that depend on cat-orientation-in-the-photographer's-frame.
- It costs nothing in storage and almost nothing in computation — the transforms run in the DataLoader's CPU workers in parallel with the GPU's training step.

In [ ]:
# The augmented training transform. Note the order: augment BEFORE ToTensor/Normalize,
# because the geometric transforms act on PIL images.
train_transform_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),         # crop a 32x32 from a 40x40 padded image
    transforms.RandomHorizontalFlip(p=0.5),       # 50% chance of mirroring left-right
    transforms.RandomRotation(degrees=15),        # small rotation up to ±15°
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Reload the training data, this time with augmentation applied.
# The validation and test sets keep their original eval_transform — augmentation
# is for training data only.
train_full_aug = datasets.CIFAR10(root="data", train=True, download=False, transform=train_transform_aug)
train_set_aug, _ = random_split(train_full_aug, [TRAIN_SIZE, VAL_SIZE], generator=torch.Generator().manual_seed(RNG_SEED))
train_loader_aug = DataLoader(train_set_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"Augmented training set ready: {len(train_set_aug)} samples")

### 5.1 Visualising what the network now sees

Let's look at the *same* training image, transformed eight different ways. The label stays `cat` (or whatever it is) but the pixels are quite different every time.

In [ ]:
# Pick a vivid sample to show off the augmentation
sample_idx = 4
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for i, ax in enumerate(axes.flat):
    img, lbl = train_set_aug[sample_idx]    # different random transform every access
    ax.imshow(un_normalise(img))
    ax.set_title(CIFAR_CLASSES[lbl], fontsize=10)
    ax.axis("off")
plt.suptitle("The same training image, 8 different augmentations", y=1.0)
plt.tight_layout(); plt.show()

**What to observe.** The crop position, flip state, and rotation angle change every time. Sometimes you'll see white pixels at the corners of the rotated images — that's where the rotation brought in 'empty' pixels, padded with zeros (which look white after un-normalisation). The label is unchanged across all eight: this is *the same training example*, viewed eight different ways.

## 6. Train with augmentation

Now we train the **same architecture** (`CIFARCNN`), with the **same hyperparameters** and **same number of epochs**, but on the **augmented training loader**. The only change is the input pipeline.

> ⏱ Another 10–15 minutes on CPU. Same warning as before.

In [ ]:
torch.manual_seed(RNG_SEED)
model_aug = CIFARCNN().to(DEVICE)
history_aug = train_model(
    model_aug, train_loader_aug, val_loader,
    max_epochs=MAX_EPOCHS, label="with augmentation"
)
plot_history(history_aug, "With augmentation")

## 7. Compare

Now we put both runs side by side. The story to look for: with augmentation, **the gap between training and validation loss should be much smaller** — and **validation accuracy should be higher** than the baseline.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history_baseline["train_loss"], label="baseline train", linestyle="--", color="#c0322b")
axes[0].plot(history_baseline["val_loss"], label="baseline val", color="#c0322b")
axes[0].plot(history_aug["train_loss"], label="augmented train", linestyle="--", color="#0369a1")
axes[0].plot(history_aug["val_loss"], label="augmented val", color="#0369a1")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Loss: baseline vs augmented"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_baseline["train_acc"], label="baseline train", linestyle="--", color="#c0322b")
axes[1].plot(history_baseline["val_acc"], label="baseline val", color="#c0322b")
axes[1].plot(history_aug["train_acc"], label="augmented train", linestyle="--", color="#0369a1")
axes[1].plot(history_aug["val_acc"], label="augmented val", color="#0369a1")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
axes[1].set_title("Accuracy: baseline vs augmented"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Baseline   final val acc: {history_baseline['val_acc'][-1]:.4f}")
print(f"Augmented  final val acc: {history_aug['val_acc'][-1]:.4f}")
print(f"Δ = {(history_aug['val_acc'][-1] - history_baseline['val_acc'][-1])*100:+.2f} percentage points")

## 8. Test set evaluation

Now we touch the test set — for the first and *only* time. We evaluate both models so we have an honest comparison.

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            preds = model(xb).argmax(dim=1).cpu().numpy()
            all_preds.append(preds); all_labels.append(yb.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)


y_pred_b, y_true = evaluate(model_baseline, test_loader)
y_pred_a, _ = evaluate(model_aug, test_loader)

print(f"Baseline test accuracy:    {(y_pred_b == y_true).mean():.4f}")
print(f"Augmented test accuracy:   {(y_pred_a == y_true).mean():.4f}")
print()
print("Augmented model — full classification report:")
print(classification_report(y_true, y_pred_a, target_names=CIFAR_CLASSES, digits=4))

In [ ]:
# Confusion matrix using class names, not integer labels.
cm = confusion_matrix(y_true, y_pred_a)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=CIFAR_CLASSES, yticklabels=CIFAR_CLASSES, ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Confusion matrix — augmented model on CIFAR-10 test set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

**What to observe.** The diagonal dominates, but check the brightest off-diagonal cells. You'll typically see classic CIFAR-10 confusions: **cat ↔ dog**, **deer ↔ horse**, **airplane ↔ ship**. These are *semantically reasonable* confusions — a small ambiguous cat photo really can look like a small dog photo. A model that confused cats with airplanes would be a much bigger concern.

## 9. Predict on a single image

Same pattern as Lab 3: grab one image from the test set, add a batch dimension with `unsqueeze(0)`, push it through the model. We'll also show the model's full probability distribution over the 10 classes so you can see how confident it is.

In [ ]:
# Try changing this index and re-running to see other predictions.
sample_idx = 16
single_x, single_y = test_set[sample_idx]
batched = single_x.unsqueeze(0).to(DEVICE)

model_aug.eval()
with torch.no_grad():
    logits = model_aug(batched)
    probs = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    pred = int(probs.argmax())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].imshow(un_normalise(single_x))
axes[0].set_title(f"True: {CIFAR_CLASSES[single_y]}  ·  Predicted: {CIFAR_CLASSES[pred]}")
axes[0].axis("off")

bars = axes[1].bar(range(10), probs, color="#9ca3af")
bars[pred].set_color("#0369a1")
if single_y != pred:
    bars[single_y].set_color("#c0322b")
axes[1].set_xticks(range(10))
axes[1].set_xticklabels(CIFAR_CLASSES, rotation=45, ha="right")
axes[1].set_ylabel("probability"); axes[1].set_title("Predicted class probabilities")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

---

## 10. Exercise 1 — augmentation strength

Do **all three** of the following. Each one re-trains for 10 epochs, so allow ~12 minutes per part. **You can reduce `MAX_EPOCHS` to 5 if time is tight — the *trend* will still be clear, just the numbers will be lower.**

**(a) No augmentation, longer training.** Take the baseline model (no augmentation) but train it for **15 epochs**. Does the extra training time help, or does it overfit further? Plot the loss curves. What does this tell you about *training longer* vs *training smarter* as a strategy?

**(b) Heavy augmentation.** Build a new training transform that adds `transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)` and increases the rotation to ±25°. Retrain. Does heavier augmentation help further, or does it start to hurt? Why might too-aggressive augmentation be counter-productive on CIFAR-10 at 32×32 resolution?

**(c) Inappropriate augmentation.** Build a training transform that includes `transforms.RandomVerticalFlip(p=0.5)`. Train with it. Test accuracy? Why would vertical flip be a *terrible* idea for CIFAR-10? (Hint: think about the class list.)

In [ ]:
# Your code for Exercise 1 (a), (b), (c) here.


*Your written observations for Exercise 1:*

(a) 

(b) 

(c) 

## 11. Exercise 2 — diagnose mistakes

Do **all three** of the following. These do *not* require any retraining — they work with the already-trained `model_aug`.

**(a) Per-class accuracy.** Compute the augmented model's accuracy *separately for each of the 10 CIFAR classes*. Sort from best to worst. Are the easiest and hardest classes what you'd expect? (Hint: think about within-class visual variation.)

**(b) Most confident mistakes.** Find the 6 test images where the model is *most confident in the wrong class* — i.e. where `probs[predicted] > 0.9` but `predicted != true`. Display the images with their true and predicted labels. What might be causing these particular mistakes?

**(c) Least confident correct predictions.** Find 6 test images where the model got the answer right but only barely — where the top probability is, say, less than 0.4. These are the model's near-misses, where a small perturbation might flip the prediction. What do these images have in common?

In [ ]:
# Your code for Exercise 2 (a), (b), (c) here.


*Your written observations for Exercise 2:*

(a) 

(b) 

(c) 

## 12. Exercise 3 — your turn to design

**Pick *one* of the following and complete it fully.** This exercise is more open-ended than the previous two — there is no single correct answer.

**Option A — Add weight decay.** Add `weight_decay=1e-4` to the Adam optimizer (this is L2 regularisation on the weights). Retrain the *baseline* (no augmentation) model with weight decay. Does weight decay alone — without augmentation — close the train/val gap? Compare against both the original baseline and the augmented model. Is weight decay a *substitute* for augmentation, or a *complement*?

**Option B — Deeper network.** Add a *fourth* conv block (`Conv2d(128 → 256, 3×3) + BN + Pool`) to the architecture. You'll need to recompute the input size of the first Linear layer (it will be `256 * 2 * 2 = 1024`). Train this deeper model **with augmentation**. Does the extra depth help, or are we already near the limit for this dataset size?

**Option C — Compare optimizers.** Train the augmented model three ways: with `SGD(lr=0.01, momentum=0.9)`, `Adam(lr=1e-3)`, and `AdamW(lr=1e-3, weight_decay=1e-4)`. Plot all three validation accuracy curves on the same axes. Which is fastest to converge? Which reaches the best final accuracy? Are the rankings what you expected from theory?

Whichever option you pick, write a 3–5 sentence reflection on what you found and what it tells you about the design choices in deep learning.

In [ ]:
# Your code for Exercise 3 (Option A, B, or C) here.


*Which option did you pick? Why?*


*Your written reflection:*



---

## 13. Reflection questions

Answer in the markdown cells below. Aim for 2–4 sentences per question.

**Q1.** State, in plain English and in one sentence, what *overfitting* is and how you can detect it from training and validation loss curves.

**Q2.** Explain why data augmentation is applied to the *training* set only. What would go wrong if you applied the same augmentations to your test set?

**Q3.** Augmentation creates *infinitely many* training views per image. Why doesn't this make your model perfect? What is the limiting factor — the number of base images, or the diversity of plausible augmentations, or something else?

**Q4.** Choosing augmentations is dataset-specific. Give one concrete example of an augmentation that is fine on CIFAR-10 but would be **harmful** on a different image dataset of your choice (e.g. MNIST, an OCR dataset, a medical imaging dataset). Justify your choice.

**Q5.** Imagine you're handed a CNN training run that *isn't* overfitting — train and validation loss track each other closely throughout. Is this always a good thing? What might it indicate about your model or your dataset?

*Your answers:*

**A1.** 

**A2.** 

**A3.** 

**A4.** 

**A5.** 

---

## What's next

In **Lab 5** we will tackle a question every real-world deep learning project starts with: *how do you train a model on a folder of image files?* All our datasets so far have been packaged and tidied by `torchvision`. Lab 5 introduces the `ImageFolder` pattern for loading custom image data from disk, working on a medical imaging task: classifying microscope images of red blood cells as malaria-infected or healthy.

Before leaving today, make sure:

- [ ] You have completed Exercises 1, 2, and 3 (your pick of Option A, B, or C)
- [ ] You have answered the reflection questions
- [ ] Your notebook runs **top to bottom without errors** (*Kernel → Restart and Run All*)
- [ ] You have saved your work — the `labs/` folder is volume-mounted on your host